# 04D — Portable Colab setup and experiments

Default: **non-training smoke test**. Open in VS Code and connect to a Colab GPU kernel. The notebook file can be local, but Python and dependencies execute remotely. Upload the files in `docs/portability/drive_migration.csv` first. Never run 00–03B to repair a missing cache.

Set your GitHub URL below after publishing the initial repository. No repository is pushed by this task.

In [ ]:
from pathlib import Path
import os, subprocess, sys
from google.colab import drive
drive.mount("/content/drive")

REPO_URL = ""  # Your GitHub HTTPS clone URL; do not embed credentials.
assert REPO_URL.startswith("https://github.com/") and "@" not in REPO_URL, "Set your GitHub HTTPS URL"
repo = Path(os.environ.get("EDGEAI_REPO_ROOT", str(Path.cwd() / "edge-ai"))).resolve()
if not repo.exists():
    subprocess.run(["git", "clone", REPO_URL, str(repo)], check=True)
else:
    origin = subprocess.check_output(["git", "remote", "get-url", "origin"], cwd=repo, text=True).strip()
    assert origin == REPO_URL, "Existing checkout has a different origin"
    assert not subprocess.check_output(["git", "status", "--porcelain"], cwd=repo, text=True).strip(), "Preserve uncommitted runtime changes before pulling"
    subprocess.run(["git", "pull", "--ff-only"], cwd=repo, check=True)
os.environ["EDGEAI_REPO_ROOT"] = str(repo)
os.chdir(repo)
sys.path.insert(0, str(repo / "tools"))
print("Remote checkout:", repo)


Install the pinned cache-consumer environment in the connected runtime. The smoke test runs in a fresh child Python process, so packages previously imported in the notebook kernel do not hide an environment mismatch. Private GitHub authentication must be configured interactively; keep credentials out of this notebook.

In [ ]:
from colab_bootstrap import configure
paths = configure(install=True)


Verify all frozen identities and eight saved baseline runs. This reads saved predictions and hashes; it does not evaluate a trained model on TEST again or regenerate any data.

In [ ]:
subprocess.run([sys.executable, "-B", "tools/verify_portability.py"], cwd=repo, check=True)


Verify `torch.cuda.is_available()`, show GPU name, then perform eight synthetic forward passes. No optimizer is constructed and no scientific experiment starts.

In [ ]:
subprocess.run([sys.executable, "-c", "import torch; print('CUDA:', torch.cuda.is_available()); assert torch.cuda.is_available(); print('GPU:', torch.cuda.get_device_name(0))"], check=True)
subprocess.run([sys.executable, "-B", "tools/experiment_runner.py", "smoke", "--device", "cuda"], cwd=repo, check=True)
print("Future outputs:", paths.EXPERIMENT_ROOT / "<experiment_id>")


## Future runs (disabled)

Copy `configs/experiments/baseline_reuse.json` to a new config locally, choose a never-used `exp_...` ID, commit and push, then pull here. The current runner deliberately supports only the accepted CUDA protocol and four existing architectures. New scientific options need separately reviewed implementations. Changes to waveform/segmentation/features belong in a new versioned `derived/` location. Existing experiments cannot be resumed or overwritten implicitly.

In [ ]:
RUN_TRAINING = False
EXPERIMENT_CONFIG = "configs/experiments/baseline_reuse.json"
if RUN_TRAINING:
    subprocess.run([sys.executable, "-B", "tools/experiment_runner.py", "train", "--config", EXPERIMENT_CONFIG], cwd=repo, check=True)
else:
    print("Training disabled. Infrastructure smoke test only.")
